# Day 14 — Semantic Search with FAISS vs Keyword Search
Semantic search (local `all-MiniLM-L6-v2` embeddings + FAISS) compared against a word-overlap keyword baseline on the same 50-document corpus. Same free local model as Day 13; the task names OpenAI's text-embedding-3-small, and the FAISS code is identical for either model (only the vector size differs).

In [1]:
!pip install -q faiss-cpu sentence-transformers numpy pandas

In [2]:
import os, re, time
import numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer

MODEL = "all-MiniLM-L6-v2"      # same free local model as Day 13 (no API key needed)
model = SentenceTransformer(MODEL)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 1. Corpus (50 short documents, 5 topics)

In [3]:
corpus = [
    'Regular aerobic exercise strengthens the heart and lowers blood pressure.',
    'Sleeping seven to nine hours each night improves memory and mood.',
    'Vitamin B12 deficiency can cause fatigue and numbness in the hands.',
    'Drinking enough water helps kidneys flush out waste from the body.',
    'Stretching before a run reduces the risk of muscle strains.',
    'A diet rich in fiber supports healthy digestion.',
    'Meditation for ten minutes a day can reduce stress levels.',
    'Doctors recommend annual checkups to catch illnesses early.',
    'Physical therapy helps patients recover mobility after knee surgery.',
    'Sunscreen with SPF 50 protects skin from harmful UV rays.',
    'The HTTP 404 error means the requested web page could not be found.',
    'TLS 1.3 reduces handshake latency compared with earlier versions.',
    'Python lists are mutable while tuples cannot be changed after creation.',
    'A neural network learns by adjusting weights through backpropagation.',
    'Laptop batteries degrade faster when kept at full charge in high heat.',
    'Git branches let developers work on features without affecting the main code.',
    'Cloud servers can scale up automatically when traffic spikes.',
    'Two-factor authentication adds a second step to protect your account.',
    'Kubernetes orchestrates containers across a cluster of machines.',
    'SQL joins combine rows from two tables based on a related column.',
    'Smartphones lose signal in elevators because metal blocks radio waves.',
    'The GPT-4 model can process both text and images.',
    'Simmer the tomato sauce for thirty minutes to deepen the flavor.',
    'Rest the steak for five minutes so the juices redistribute.',
    'Yeast needs warm water to activate before baking bread.',
    'Cast iron pans should be seasoned with oil to prevent rust.',
    'Whisk egg whites until stiff peaks form for a light meringue.',
    'Marinating chicken in yogurt makes the meat tender.',
    'Roasting vegetables at high heat caramelizes their natural sugars.',
    'Sharpen kitchen knives regularly because dull blades slip more easily.',
    'Basmati rice should be rinsed before cooking to remove excess starch.',
    'Fresh basil loses its aroma quickly when heated for too long.',
    'An emergency fund should cover three to six months of expenses.',
    'Compound interest grows savings faster the longer money stays invested.',
    'Index funds offer low-cost diversification across the stock market.',
    'Inflation reduces the purchasing power of cash over time.',
    'A credit score above 750 usually qualifies for the best loan rates.',
    'Budgeting apps help track monthly spending categories.',
    'The RBI repo rate influences the interest on home loans.',
    'Diversifying investments lowers the risk of large losses.',
    'Paying off high-interest debt first saves money in the long run.',
    'Book flights on a Tuesday afternoon to find cheaper tickets.',
    'Carry a copy of your passport when exploring a foreign city.',
    'Kashmir in spring is famous for blooming tulip gardens.',
    'Travel insurance covers cancelled trips and medical emergencies abroad.',
    'Pack light with a carry-on bag to avoid baggage fees.',
    'Public trains are often the cheapest way to cross Europe.',
    'Altitude sickness can affect hikers above three thousand meters.',
    'Learning a few local phrases helps travelers connect with residents.',
    'Jet lag eases faster if you adjust your sleep schedule before departure.',
]
print(len(corpus), 'documents')

50 documents


## 2. Embed the corpus → NumPy float32
FAISS needs float32 arrays. I normalise vectors to length 1 so that squared-L2 distance maps cleanly to cosine similarity.

In [4]:
def embed(texts):
    vecs = model.encode(list(texts), normalize_embeddings=True)
    return np.asarray(vecs, dtype="float32")

t0 = time.time()
doc_vectors = embed(corpus)
embed_time = time.time() - t0
np.save("corpus_vectors.npy", doc_vectors)

print("shape:", doc_vectors.shape, "| dtype:", doc_vectors.dtype, "| time: %.2fs" % embed_time)

shape: (50, 384) | dtype: float32 | time: 0.64s


## 3. FAISS flat index

In [5]:
dim = doc_vectors.shape[1]          # 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatL2(dim)      # exact (brute-force) L2 search
index.add(doc_vectors)

print("Index size:", index.ntotal, "| documents:", len(corpus))
assert index.ntotal == len(corpus), "Some documents were not stored!"

Index size: 50 | documents: 50


## 4. `semantic_search` and `keyword_search`
Vectors are unit length, so squared-L2 distance = 2 − 2·cosine. I print both.

In [6]:
def semantic_search(query, top_k=3):
    q = embed([query])
    D, I = index.search(q, top_k)
    return [(int(i), float(d), 1 - float(d) / 2) for d, i in zip(D[0], I[0])]   # (doc_id, L2 dist, cosine)

STOP = {"a","an","the","is","are","in","on","of","to","for","and","or","my","i","how","do","does","what","with","can","it","by","at","as","be","when","from"}
def tokens(text):
    return {w for w in re.findall(r"\w+", text.lower()) if w not in STOP}

def keyword_search(query, corpus, top_k=3):
    q = tokens(query)
    scored = []
    for i, doc in enumerate(corpus):
        overlap = len(q & tokens(doc))
        if overlap:
            scored.append((i, overlap / len(q)))       # fraction of query words found
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

print(semantic_search("cardiovascular workout benefits", 3))
print(keyword_search("cardiovascular workout benefits", corpus, 3))

[(0, 0.6393437385559082, 0.6803281307220459), (4, 1.0908997058868408, 0.4545501470565796), (6, 1.3977895975112915, 0.30110520124435425)]
[]


## 5. Ten test queries, side by side
`expected` = the document I consider the correct answer, so I can measure the rank each method gave it.

In [7]:
test_queries = [
    ('cardiovascular workout benefits', 0),
    ('my phone has no reception in the lift', 20),
    ('cheap airfare tips', 41),
    ('HTTP 404', 10),
    ('TLS 1.3', 11),
    ('Vitamin B12', 2),
    ('RBI repo rate', 38),
    ('Kashmir tulip', 43),
    ('tenderise meat', 27),
    ('protect my account online', 17),
]

In [8]:
rows = []
for q, exp in test_queries:
    sem = semantic_search(q, 3)
    kw  = keyword_search(q, corpus, 3)
    sem_ids = [s[0] for s in sem]; kw_ids = [k[0] for k in kw]

    print("=" * 90); print("QUERY:", q, f"(expected doc {exp})")
    print(" SEMANTIC:")
    for i, d, c in sem: print(f"   [{i:2d}] cos={c:.3f} L2={d:.3f}  {corpus[i]}")
    print(" KEYWORD :")
    if not kw: print("   (no matching words)")
    for i, s in kw: print(f"   [{i:2d}] overlap={s:.2f}  {corpus[i]}")

    rows.append({
        "query": q, "expected": exp,
        "semantic_top1": sem_ids[0], "keyword_top1": kw_ids[0] if kw else None,
        "semantic_hit": exp in sem_ids[:1], "keyword_hit": exp in kw_ids[:1],
    })

results = pd.DataFrame(rows)
results

QUERY: cardiovascular workout benefits (expected doc 0)
 SEMANTIC:
   [ 0] cos=0.680 L2=0.639  Regular aerobic exercise strengthens the heart and lowers blood pressure.
   [ 4] cos=0.455 L2=1.091  Stretching before a run reduces the risk of muscle strains.
   [ 6] cos=0.301 L2=1.398  Meditation for ten minutes a day can reduce stress levels.
 KEYWORD :
   (no matching words)
QUERY: my phone has no reception in the lift (expected doc 20)
 SEMANTIC:
   [20] cos=0.506 L2=0.988  Smartphones lose signal in elevators because metal blocks radio waves.
   [47] cos=0.188 L2=1.624  Altitude sickness can affect hikers above three thousand meters.
   [ 8] cos=0.159 L2=1.683  Physical therapy helps patients recover mobility after knee surgery.
 KEYWORD :
   (no matching words)
QUERY: cheap airfare tips (expected doc 41)
 SEMANTIC:
   [41] cos=0.597 L2=0.806  Book flights on a Tuesday afternoon to find cheaper tickets.
   [46] cos=0.441 L2=1.118  Public trains are often the cheapest way to cross Eur

,query,expected,semantic_top1,keyword_top1,semantic_hit,keyword_hit
0,cardiovascular workout benefits,0,0,NaN,True,False
1,my phone has no reception in the lift,20,20,NaN,True,False
2,cheap airfare tips,41,41,NaN,True,False
3,HTTP 404,10,10,10.0,True,True
4,TLS 1.3,11,11,11.0,True,True
5,Vitamin B12,2,2,2.0,True,True
6,RBI repo rate,38,38,38.0,True,True
7,Kashmir tulip,43,43,43.0,True,True
8,tenderise meat,27,27,27.0,True,True
9,protect my account online,17,17,17.0,True,True


## 6. Which queries went which way?

In [9]:
sem_wins = results[results.semantic_hit & ~results.keyword_hit]
kw_wins  = results[results.keyword_hit & ~results.semantic_hit]
print("SEMANTIC finds it, KEYWORD fails (vocabulary mismatch):"); display(sem_wins[["query"]])
print("KEYWORD is correct and semantic is not (or no better):"); display(kw_wins[["query"]])
both = results[results.semantic_hit & results.keyword_hit]
print("Both correct:"); display(both[["query"]])

SEMANTIC finds it, KEYWORD fails (vocabulary mismatch):


,query
0,cardiovascular workout benefits
1,my phone has no reception in the lift
2,cheap airfare tips


KEYWORD is correct and semantic is not (or no better):


,query


Both correct:


,query
3,HTTP 404
4,TLS 1.3
5,Vitamin B12
6,RBI repo rate
7,Kashmir tulip
8,tenderise meat
9,protect my account online


> **Note for submission:** if fewer than 3 queries land in the keyword-wins table (modern embeddings are strong on short exact tokens too), also compare the *similarity margin* — e.g. for `HTTP 404` look at how close the other tech documents score in the semantic list vs the clean exact-match in keyword. Write what you actually observe below.

## 7. Latency and cost

In [10]:
import timeit
q = "cheap airfare tips"
q_vec = embed([q])
t_embed = timeit.timeit(lambda: embed([q]), number=20) / 20
t_faiss = timeit.timeit(lambda: index.search(q_vec, 3), number=1000) / 1000
t_kw    = timeit.timeit(lambda: keyword_search(q, corpus, 3), number=1000) / 1000

print(f"Semantic: query embedding (local model) {t_embed*1000:.1f} ms + FAISS search {t_faiss*1000:.3f} ms")
print(f"Keyword : {t_kw*1000:.3f} ms")

# Cost: local model = $0. For comparison, OpenAI text-embedding-3-small is ~$0.02 per 1M tokens (verify on their pricing page)
tokens_est = sum(len(d.split()) for d in corpus) * 1.3
print(f"Actual cost (local model): $0.00")
print(f"Corpus ~{tokens_est:.0f} tokens -> ~${tokens_est/1e6*0.02:.6f} if embedded with OpenAI")
print(f"1M docs x 100 tokens = 100M tokens -> ~${100e6/1e6*0.02:.2f} with OpenAI; vectors = {1_000_000*384*4/1e9:.1f} GB (384-dim) or {1_000_000*1536*4/1e9:.1f} GB (1536-dim OpenAI)")

Semantic: query embedding (local model) 18.4 ms + FAISS search 0.015 ms
Keyword : 0.382 ms
Actual cost (local model): $0.00
Corpus ~650 tokens -> ~$0.000013 if embedded with OpenAI
1M docs x 100 tokens = 100M tokens -> ~$2.00 with OpenAI; vectors = 1.5 GB (384-dim) or 6.1 GB (1536-dim OpenAI)


## 8. Trade-offs

**When semantic search is better:** when users describe what they want in their own words.
"my phone has no reception in the lift" found the smartphone/elevator document (cos 0.51)
even though no words matched. Keyword search returned nothing for all 3 such queries.

**When keyword search is better:** for exact identifiers like "HTTP 404", "TLS 1.3" and
"Vitamin B12". Both methods found the right top result, but keyword returned only the one
matching document. Semantic always returns top_k, so it added unrelated documents
(tomato sauce, steak) with scores of about 0.1. A similarity threshold would fix this.

**Latency:** embedding the query took 18.4 ms and the FAISS search 0.015 ms. Keyword took 0.38 ms.
The model call is the slow part of semantic search.

**Cost:** $0 with my local model. OpenAI embeddings would cost about $2 per 100M tokens.

**Scale:** 1M documents take 1.5 GB at 384 dimensions (6.1 GB at 1536). A flat index
scans every vector, so large systems use approximate indexes (HNSW or IVF).

**Conclusion:** use a hybrid in production: keyword for exact terms, semantic for
meaning, with a score threshold.
